In [1]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
#set dataset paths
DATA_DIR = Path("C:/Users/sreer/OneDrive/Desktop/Sasi Deepika/Infotact/cleaned_data")

TRANSACTIONS_FILE = DATA_DIR / "transactions_clean.csv"
PROMOTIONS_FILE = DATA_DIR / "promotions_clean.csv"

print("Current notebook directory:", Path.cwd().resolve())
print("Transactions file exists:", TRANSACTIONS_FILE.exists())
print("Promotions file exists:", PROMOTIONS_FILE.exists())

if not TRANSACTIONS_FILE.exists():
    raise FileNotFoundError(
        "Check the path to transactions_clean.csv"
    )

if not PROMOTIONS_FILE.exists():
    raise FileNotFoundError(
        "Check the path to promotions_clean.csv"
    )

Current notebook directory: C:\Users\sreer\Downloads
Transactions file exists: True
Promotions file exists: True


In [3]:
#Load the sample data
transactions_sample = pd.read_csv(
    TRANSACTIONS_FILE,
    nrows=10
)

promotions_sample = pd.read_csv(
    PROMOTIONS_FILE,
    nrows=10
)

print("TRANSACTIONS DATA")
display(transactions_sample)

print("PROMOTIONS DATA")
display(promotions_sample)

TRANSACTIONS DATA


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.00,0.00
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.00,0.00
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.00,0.00
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.00,0.00
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.00,0.00
5,2375,26984851516,1,826249,2,1.98,364,-0.60,1642,1,0.00,0.00
6,2375,26984851516,1,1043142,1,1.57,364,-0.68,1642,1,0.00,0.00
7,2375,26984851516,1,1085983,1,2.99,364,-0.40,1642,1,0.00,0.00
8,2375,26984851516,1,1102651,1,1.89,364,0.00,1642,1,0.00,0.00
9,2375,26984851516,1,6423775,1,2.00,364,-0.79,1642,1,0.00,0.00


PROMOTIONS DATA


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A
3,26190,292,70,0,A
4,26190,293,70,0,A
5,26190,295,70,0,A
6,26190,296,70,0,A
7,26190,297,70,0,A
8,26190,298,70,0,A
9,26190,299,70,0,A


In [4]:
#Check columns and data types
print("Transaction columns:")
print(transactions_sample.columns.tolist())

print("\nTransaction data types:")
display(transactions_sample.dtypes.rename("Data Type").to_frame())

print("\nPromotion columns:")
print(promotions_sample.columns.tolist())

print("\nPromotion data types:")
display(promotions_sample.dtypes.rename("Data Type").to_frame())

Transaction columns:
['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC']

Transaction data types:


,Data Type
household_key,int64
BASKET_ID,int64
DAY,int64
PRODUCT_ID,int64
QUANTITY,int64
SALES_VALUE,float64
STORE_ID,int64
RETAIL_DISC,float64
TRANS_TIME,int64
WEEK_NO,int64



Promotion columns:
['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']

Promotion data types:


,Data Type
PRODUCT_ID,int64
STORE_ID,int64
WEEK_NO,int64
display,int64
mailer,object


In [5]:
#Create a chunk-reading function
def read_csv_chunks(file_path, chunksize=250_000):
    """Read a CSV in chunks to reduce memory usage."""
    yield from pd.read_csv(file_path, chunksize=chunksize)

In [6]:
#Calculate transaction dataset statistics
transaction_rows = 0

unique_households = set()
unique_baskets = set()
unique_products = set()
unique_stores = set()
unique_weeks = set()

missing_counts = {}
duplicate_rows = 0

zero_quantity = 0
zero_sales = 0
negative_sales = 0
quantity_above_100 = 0

for chunk in read_csv_chunks(TRANSACTIONS_FILE):

    transaction_rows += len(chunk)

    # Missing values
    for col in chunk.columns:
        missing_counts[col] = (
            missing_counts.get(col, 0)
            + int(chunk[col].isna().sum())
        )

    # Duplicate rows within each chunk (initial check)
    duplicate_rows += int(chunk.duplicated().sum())

    # Unique entities
    unique_households.update(
        chunk["household_key"].dropna().unique()
    )
    unique_baskets.update(
        chunk["BASKET_ID"].dropna().unique()
    )
    unique_products.update(
        chunk["PRODUCT_ID"].dropna().unique()
    )
    unique_stores.update(
        chunk["STORE_ID"].dropna().unique()
    )
    unique_weeks.update(
        chunk["WEEK_NO"].dropna().unique()
    )

    # Basic data quality checks
    zero_quantity += int((chunk["QUANTITY"] == 0).sum())
    zero_sales += int((chunk["SALES_VALUE"] == 0).sum())
    negative_sales += int((chunk["SALES_VALUE"] < 0).sum())
    quantity_above_100 += int((chunk["QUANTITY"] > 100).sum())

print("Transaction dataset analysis completed.")

Transaction dataset analysis completed.


In [7]:
#Display transaction summary
transaction_summary = pd.DataFrame({
    "Metric": [
        "Total transaction rows",
        "Unique households",
        "Unique baskets",
        "Unique products",
        "Unique stores",
        "Distinct weeks",
        "Rows with zero quantity",
        "Rows with zero sales",
        "Rows with negative sales",
        "Rows with quantity above 100",
        "Duplicate rows within chunks"
    ],
    "Value": [
        transaction_rows,
        len(unique_households),
        len(unique_baskets),
        len(unique_products),
        len(unique_stores),
        len(unique_weeks),
        zero_quantity,
        zero_sales,
        negative_sales,
        quantity_above_100,
        duplicate_rows
    ]
})

display(transaction_summary)

,Metric,Value
0,Total transaction rows,2595732
1,Unique households,2500
2,Unique baskets,276484
3,Unique products,92339
4,Unique stores,582
5,Distinct weeks,102
6,Rows with zero quantity,14466
7,Rows with zero sales,18850
8,Rows with negative sales,0
9,Rows with quantity above 100,23136


In [8]:
#Missing-value analysis
missing_df = pd.DataFrame({
    "Column": list(missing_counts.keys()),
    "Missing Values": list(missing_counts.values())
})

missing_df["Missing Percentage"] = (
    missing_df["Missing Values"] / transaction_rows * 100
)

display(
    missing_df.sort_values(
        "Missing Percentage",
        ascending=False
    )
)

,Column,Missing Values,Missing Percentage
0,household_key,0,0.00
1,BASKET_ID,0,0.00
2,DAY,0,0.00
3,PRODUCT_ID,0,0.00
4,QUANTITY,0,0.00
5,SALES_VALUE,0,0.00
6,STORE_ID,0,0.00
7,RETAIL_DISC,0,0.00
8,TRANS_TIME,0,0.00
9,WEEK_NO,0,0.00


In [9]:
#Promotion dataset overview
promotion_rows = 0
promotion_missing_counts = {}

unique_promotion_products = set()
unique_promotion_stores = set()
unique_promotion_weeks = set()

for chunk in read_csv_chunks(
    PROMOTIONS_FILE,
    chunksize=500_000
):

    promotion_rows += len(chunk)

    for col in chunk.columns:
        promotion_missing_counts[col] = (
            promotion_missing_counts.get(col, 0)
            + int(chunk[col].isna().sum())
        )

    unique_promotion_products.update(
        chunk["PRODUCT_ID"].dropna().unique()
    )
    unique_promotion_stores.update(
        chunk["STORE_ID"].dropna().unique()
    )
    unique_promotion_weeks.update(
        chunk["WEEK_NO"].dropna().unique()
    )

promotion_summary = pd.DataFrame({
    "Metric": [
        "Total promotion rows",
        "Unique products",
        "Unique stores",
        "Distinct weeks"
    ],
    "Value": [
        promotion_rows,
        len(unique_promotion_products),
        len(unique_promotion_stores),
        len(unique_promotion_weeks)
    ]
})

display(promotion_summary)

,Metric,Value
0,Total promotion rows,36786524
1,Unique products,68377
2,Unique stores,115
3,Distinct weeks,93


In [10]:
#Promotion missing values
promotion_missing_df = pd.DataFrame({
    "Column": list(promotion_missing_counts.keys()),
    "Missing Values": list(promotion_missing_counts.values())
})

promotion_missing_df["Missing Percentage"] = (
    promotion_missing_df["Missing Values"]
    / promotion_rows * 100
)

display(
    promotion_missing_df.sort_values(
        "Missing Percentage",
        ascending=False
    )
)

,Column,Missing Values,Missing Percentage
0,PRODUCT_ID,0,0.00
1,STORE_ID,0,0.00
2,WEEK_NO,0,0.00
3,display,0,0.00
4,mailer,0,0.00


In [11]:
#Statistical summary
sample_parts = []
sample_limit = 100_000
sample_count = 0

for chunk in read_csv_chunks(
    TRANSACTIONS_FILE,
    chunksize=100_000
):

    remaining = sample_limit - sample_count

    if remaining <= 0:
        break

    take = min(len(chunk), remaining)

    sample_parts.append(chunk.head(take))
    sample_count += take

transactions_sample = pd.concat(
    sample_parts,
    ignore_index=True
)

display(transactions_sample.describe().T)

,count,mean,std,min,25%,50%,75%,max
household_key,"100,000.00","1,277.93",739.55,1.00,631.00,"1,260.00","1,946.00","2,500.00"
BASKET_ID,"100,000.00","27,580,942,297.09","247,816,999.96","26,984,851,472.00","27,398,746,609.00","27,620,291,018.00","27,785,932,886.00","27,924,961,066.00"
DAY,"100,000.00",50.48,20.59,1.00,35.00,54.00,68.00,79.00
PRODUCT_ID,"100,000.00","2,065,860.03","2,502,277.41","26,691.00","909,479.00","1,013,245.00","1,110,120.75","10,364,237.00"
QUANTITY,"100,000.00",49.39,775.43,0.00,1.00,1.00,1.00,"39,365.00"
SALES_VALUE,"100,000.00",2.99,3.64,0.00,1.29,2.00,3.49,400.00
STORE_ID,"100,000.00","2,368.43","7,616.33",1.00,327.00,367.00,412.00,"32,004.00"
RETAIL_DISC,"100,000.00",-0.56,1.13,-38.97,-0.79,-0.13,0.00,0.00
TRANS_TIME,"100,000.00","1,594.55",408.29,0.00,"1,326.00","1,634.00","1,909.00","2,359.00"
WEEK_NO,"100,000.00",7.91,2.97,1.00,6.00,8.00,10.00,12.00


In [12]:
OUTPUT_DIR = Path("../outputs/eda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

transaction_summary.to_csv(
    OUTPUT_DIR / "day1_transaction_summary.csv",
    index=False
)

missing_df.to_csv(
    OUTPUT_DIR / "day1_transaction_missing_values.csv",
    index=False
)

promotion_summary.to_csv(
    OUTPUT_DIR / "day1_promotion_summary.csv",
    index=False
)

promotion_missing_df.to_csv(
    OUTPUT_DIR / "day1_promotion_missing_values.csv",
    index=False
)

print("Day 1 results saved successfully.")

Day 1 results saved successfully.
